# Pipeline Preditivo de Manutencao Industrial

Projeto de Ciencia de Dados aplicado a predicao de falhas mecanicas em equipamentos industriais a partir de leituras de sensores de temperatura, rotacao, torque e desgaste de ferramenta.

## Fase 1 - Exploracao de Dados (EDA)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

df = pd.read_csv("data/manutencao_preditiva.csv")
print("Dimensoes da base (linhas, colunas):", df.shape)
print("\n Estrutura e tipos de dados:")
df.info()

In [ ]:
df.describe().T

In [ ]:
colunas_sensores = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
    "desgaste_ferramenta_min"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for idx, col in enumerate(colunas_sensores):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[idx], color="#2b5c8f", bins=30)
    axes[idx].set_title(f"Distribuicao de {col}")
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel("Frequencia")

axes[5].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
contagem_alvo = df["falha_maquina"].value_counts()
proporcao_alvo = df["falha_maquina"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(7, 5))
barras = sns.barplot(
    x=contagem_alvo.index.astype(str),
    y=contagem_alvo.values,
    hue=contagem_alvo.index.astype(str),
    palette=["#2b5c8f", "#c93b2b"],
    legend=False,
    ax=ax
)

for idx, valor in enumerate(contagem_alvo.values):
    pct = proporcao_alvo.iloc[idx]
    ax.text(idx, valor + 150, f"{valor} ({pct:.2f}%)", ha="center", fontweight="bold")

ax.set_title("Desbalanceamento da Variavel Alvo (falha_maquina)")
ax.set_xlabel("Estado da Maquina (0 = Operacao Normal, 1 = Falha)")
ax.set_ylabel("Quantidade de Registros")
ax.set_ylim(0, 11000)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
matriz_corr = df[colunas_sensores + ["falha_maquina"]].corr(method="pearson")
sns.heatmap(
    matriz_corr,
    annot=True,
    fmt=".4f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title("Mapa de Calor - Correlacao Linear de Pearson")
plt.tight_layout()
plt.show()

### Analise Interpretativa da Exploracao de Dados

A inspecao inicial da base `data/manutencao_preditiva.csv` mostra exatamente 10.000 registros e 14 colunas. A leitura cruzada com o documento tecnico do Departamento de Engenharia define a funcao de cada atributo:
- As colunas `udi` (1 a 10.000) e `id_produto` (10.000 codigos exclusivos) funcionam apenas como chaves identificadoras de registro e serie, sem valor preditivo generalizavel.
- As colunas `falha_twf`, `falha_hdf`, `falha_pwf`, `falha_osf` e `falha_rnf` registram o motivo especifico da quebra apos a ocorrencia da falha. Conforme especificado pela engenharia, essas cinco variaveis sao registros de historico pos-evento e devem ser removidas da matriz preditora `X` para evitar vazamento direto da resposta.
- A coluna `tipo` classifica a qualidade do equipamento em tres categorias: `L` com 6.000 registros (60,00%), `M` com 2.997 registros (29,97%) e `H` com 1.003 registros (10,03%).

O `.info()` e o `.describe()` revelam que exatamente 4 colunas continuas possuem 9.500 registros preenchidos e 500 valores ausentes (`NaN`): `temperatura_ar_k`, `temperatura_processo_k`, `velocidade_rotacao_rpm` e `torque_nm`. Ja `desgaste_ferramenta_min` possui os 10.000 registros completos, variando entre 0 e 253 minutos (media de 107,95 minutos e mediana de 108,00 minutos).

A analise dos tres graficos evidencia tres comportamentos centrais para o pipeline:
1. **Distribuicao das variaveis preditoras:** As variaveis `temperatura_ar_k` (media de 300,00 K e mediana de 300,10 K), `temperatura_processo_k` (media de 310,00 K e mediana de 310,10 K) e `torque_nm` (media de 39,97 Nm e mediana de 40,10 Nm) apresentam distribuicoes simetricas proximas a normalidade. Em contrapartida, `velocidade_rotacao_rpm` apresenta forte assimetria positiva (cauda longa a direita chegando ao maximo de 2.886,0 RPM), deslocando a media (1.539,25 RPM) acima da mediana (1.504,00 RPM).
2. **Desbalanceamento severo da classe alvo:** A variavel `falha_maquina` contabiliza 9.661 registros na classe 0 (96,61% de operacao normal) contra apenas 339 registros na classe 1 (3,39% de falha mecanica). Treinar classificadores diretamente nessa proporcao enviesaria o modelo para prever sempre a classe majoritaria.
3. **Correlacoes lineares de Pearson:** Existe correlacao positiva forte de `0.8766` entre `temperatura_ar_k` e `temperatura_processo_k`, refletindo o acoplamento termico entre o ambiente da fabrica e o processo de usinagem. Observa-se tambem correlacao negativa forte de `-0.8754` entre `velocidade_rotacao_rpm` e `torque_nm`, comportamento fisico esperado em motores eletricos onde o aumento da carga de torque reduz a rotacao. Em relacao ao alvo `falha_maquina`, as maiores correlacoes individuais ocorrem com `torque_nm` (`0.1889`) e `desgaste_ferramenta_min` (`0.1054`).

## Fase 2 - Limpeza e Tratamento de Dados

In [ ]:
duplicados_totais = df.duplicated().sum()
print(f"Registros duplicados na base bruta: {duplicados_totais}")

df = df.drop_duplicates().copy()
print(f"Dimensoes apos verificacao de duplicados: {df.shape}")

print("\nContagem de valores ausentes por coluna:")
print(df[colunas_sensores].isnull().sum())

print("\nAssimetria (skewness), Media e Mediana antes da imputacao:")
for col in ["temperatura_ar_k", "temperatura_processo_k", "velocidade_rotacao_rpm", "torque_nm"]:
    skew_val = df[col].skew()
    media_val = df[col].mean()
    mediana_val = df[col].median()
    print(f"{col:24s} | skew: {skew_val:7.4f} | media: {media_val:9.4f} | mediana: {mediana_val:9.4f}")

In [ ]:
# Imputacao pela media nas variaveis com distribuicao simetrica (|skew| < 0.15)
for col_simetrica in ["temperatura_ar_k", "temperatura_processo_k", "torque_nm"]:
    df[col_simetrica] = df[col_simetrica].fillna(df[col_simetrica].mean())

# Imputacao pela mediana em velocidade_rotacao_rpm devido a forte assimetria positiva (skew = 1.9946)
df["velocidade_rotacao_rpm"] = df["velocidade_rotacao_rpm"].fillna(df["velocidade_rotacao_rpm"].median())

print("Contagem de nulos apos imputacao:")
print(df.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))

for idx, col in enumerate(colunas_sensores):
    sns.boxplot(y=df[col], ax=axes[idx], color="#4a7c59", width=0.45)
    axes[idx].set_title(col)
    axes[idx].set_ylabel("Valor")

plt.suptitle("Boxplots das Variaveis Explicativas para Inspecao de Outliers", y=1.03, fontsize=13)
plt.tight_layout()
plt.show()

for col in colunas_sensores:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    qtd_outliers = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
    print(f"{col:24s} | IQR: {iqr:6.2f} | Limites: [{lim_inf:7.2f}, {lim_sup:7.2f}] | Outliers: {qtd_outliers}")

### Justificativa do Tratamento de Nulos e Analise de Outliers

Na verificacao de integridade, `df.duplicated().sum()` confirmou 0 linhas integralmente duplicadas, mantendo os 10.000 registros. Na inspecao de valores ausentes, identificou-se que exatamente 500 registros possuiam `NaN` simultaneamente nas quatro variaveis medidas por sensores continuos (`temperatura_ar_k`, `temperatura_processo_k`, `velocidade_rotacao_rpm` e `torque_nm`). Remover 500 linhas descartaria 5,00% da base historica, pelo que se adotou a imputacao estatistica guiada pela assimetria de cada distribuicao:
- **Imputacao pela Media (`temperatura_ar_k`, `temperatura_processo_k` e `torque_nm`):** Essas tres variaveis possuem coeficientes de assimetria praticamente nulos (`0.1149`, `0.0156` e `-0.0079`, respectivamente), com medias e medianas quase identicas (`300.0022 K` vs `300.1000 K`, `310.0009 K` vs `310.1000 K`, e `39.9742 Nm` vs `40.1000 Nm`). Nesses casos, a media aritmetica preserva o centro de massa da distribuicao sem distorcao.
- **Imputacao pela Mediana (`velocidade_rotacao_rpm`):** Esta coluna apresenta coeficiente de assimetria de `1.9946` e cauda superior longa ate `2886.0 RPM`. Como a media (`1539.2453 RPM`) e puxada para cima pelos valores extremos, utilizou-se a mediana (`1504.0000 RPM`), que e uma medida de tendencia central robusta a outliers.

A inspecao pelos boxplots e pelo criterio do intervalo interquartil (IQR) apos a imputacao revelou:
- `velocidade_rotacao_rpm`: 465 outliers acima do limite superior de `1874.50 RPM`.
- `torque_nm`: 102 outliers fora do intervalo `[14.40 Nm, 65.60 Nm]`.
- `temperatura_processo_k`: 2 outliers abaixo do limite inferior de `305.75 K`.
- `temperatura_ar_k` e `desgaste_ferramenta_min`: 0 outliers.

Por regra de negocio de manutencao preditiva, esses outliers **nao devem ser removidos**, pois picos de torque acima de `65.60 Nm` e rotacoes extremas acima de `1874.50 RPM` nao sao erros de digitacao, mas sim os regimes de sobrecarga fisica e potencia que antecedem a quebra real da maquina.

## Fase 3 - Engenharia de Recursos (Feature Engineering)

In [ ]:
# A potencia mecanica instantanea e proporcional ao produto entre rotacao e torque
df["potencia"] = df["velocidade_rotacao_rpm"] * df["torque_nm"]

nulos_potencia = df["potencia"].isnull().sum()
assert nulos_potencia == 0, "Erro: existem valores ausentes na coluna potencia"
print(f"Total de valores NaN na nova coluna 'potencia': {nulos_potencia}")
print("\nResumo estatistico da variavel 'potencia':")
print(df["potencia"].describe())

### Explicacao Tecnica da Variavel `potencia`

Na mecanica rotacional, a potencia exigida do motor e proporcional ao produto da velocidade angular pela forca de torcao (`velocidade_rotacao_rpm * torque_nm`). Na Fase 1, observamos que `velocidade_rotacao_rpm` e `torque_nm` possuem correlacao negativa forte (`-0.8754`), o que significa que analisar cada uma isoladamente em cortes ortogonais dificulta a deteccao de falhas por falta ou excesso de potencia (`falha_pwf`).

Como a coluna `potencia` foi calculada apos a imputacao dos 500 valores ausentes na Fase 2, a validacao confirmou `0` registros `NaN` nos 10.000 exemplos. A nova variavel apresenta media de `59961.21`, mediana de `60121.15`, desvio padrao de `9951.11` e amplitude entre `10966.80` e `99980.40`, sintetizando o esforco energetico combinado do equipamento para os algoritmos de classificacao.

## Fase 4 - Divisao de Treino/Teste e Balanceamento

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Codificacao ordinal da categoria de qualidade do equipamento (Low=0, Medium=1, High=2)
df_modelo = df.copy()
df_modelo["tipo"] = df_modelo["tipo"].map({"L": 0, "M": 1, "H": 2})

# Remocao de identificadores e colunas pos-evento que causariam vazamento direto do alvo
colunas_remover = [
    "udi",
    "id_produto",
    "falha_maquina",
    "falha_twf",
    "falha_hdf",
    "falha_pwf",
    "falha_osf",
    "falha_rnf"
]

X = df_modelo.drop(columns=colunas_remover)
y = df_modelo["falha_maquina"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Dimensoes do treino antes do SMOTE: {X_train.shape}")
print(f"Distribuicao de y_train antes do SMOTE:\n{y_train.value_counts()}\n")
print(f"Dimensoes do teste (intocado): {X_test.shape}")
print(f"Distribuicao de y_test (intocado):\n{y_test.value_counts()}\n")

# Balanceamento aplicado exclusivamente nos dados de treino para evitar Data Leakage
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"Dimensoes do treino apos SMOTE: {X_train_bal.shape}")
print(f"Distribuicao de y_train_bal apos SMOTE:\n{y_train_bal.value_counts()}")

### Prevencao de Data Leakage na Divisao e no Balanceamento

Para preparar a matriz preditora `X` (7 atributos: `tipo`, `temperatura_ar_k`, `temperatura_processo_k`, `velocidade_rotacao_rpm`, `torque_nm`, `desgaste_ferramenta_min` e `potencia`), foram descartados os identificadores (`udi`, `id_produto`) e os cinco indicadores especificos de falha (`falha_twf` a `falha_rnf`), pois estes so existem no banco apos a maquina quebrar.

A separacao em 80% para treino (`8.000` registros) e 20% para teste (`2.000` registros) utilizou `stratify=y` e `random_state=42`, garantindo exatamente a mesma proporcao original de `3.39%` de falhas em ambos os subconjuntos: `7.729` normais e `271` falhas no treino, e `1.932` normais e `68` falhas no teste.

O algoritmo `SMOTE` foi aplicado **exclusivamente sobre `X_train` e `y_train`**, elevando a classe minoritaria de `271` para `7.729` exemplos e totalizando `15.458` registros balanceados (50% de cada classe) para o aprendizado dos modelos. Caso o `SMOTE` fosse executado antes do `train_test_split`, amostras sinteticas no treino seriam geradas por interpolacao de vizinhos que cairiam no conjunto de teste, contaminando a avaliacao com informacao privilegiada (Data Leakage) e produzindo metricas irreais que nao se sustentam em producao. O conjunto de teste permaneceu intacto com os 2.000 registros reais.

## Fase 5 - Escalonamento de Variaveis

In [ ]:
from sklearn.preprocessing import StandardScaler

colunas_continuas = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
    "desgaste_ferramenta_min",
    "potencia"
]

# O KNN calcula distancias euclidianas e exige variaveis continuas na mesma escala (media 0, desvio 1)
# A Arvore de Decisao particiona um atributo por vez por impureza de Gini, sendo imune a escala
scaler = StandardScaler()

X_train_knn = X_train_bal.copy()
X_test_knn = X_test.copy()

X_train_knn[colunas_continuas] = scaler.fit_transform(X_train_bal[colunas_continuas])
X_test_knn[colunas_continuas] = scaler.transform(X_test[colunas_continuas])

print("Medias das variaveis continuas no treino escalonado (KNN):")
print(X_train_knn[colunas_continuas].mean().round(4))
print("\nDesvios padrao das variaveis continuas no treino escalonado (KNN):")
print(X_train_knn[colunas_continuas].std().round(4))

### Justificativa do Escalonamento para o KNN e Imunidade da Arvore de Decisao

O escalonamento via `StandardScaler` foi ajustado com `fit_transform` apenas nas 6 variaveis continuas do conjunto de treino balanceado (`X_train_bal`) e aplicado com `transform` no conjunto de teste (`X_test`), impedindo que a media e o desvio padrao do teste vazem para o treinamento.

Essa diferenciacao entre os dois algoritmos decorre da matematica interna de cada classificador:
- **Dependencia de escala no KNN:** O K-Nearest Neighbors classifica cada amostra calculando a distancia geometrica (como a distancia Euclidiana) entre vetores no espaco multidimensional. Sem escalonamento, a variavel `potencia` (que varia de `10.966,80` a `99.980,40`) possui magnitude milhares de vezes superior a `temperatura_ar_k` (`295,30 K` a `304,50 K`) e `torque_nm` (`3,80 Nm` a `76,60 Nm`), fazendo com que o calculo de distancia seja completamente dominado pela potencia. Com o `StandardScaler`, todas as variaveis continuas passam a ter media `0.0000` e desvio padrao `1.0000`, contribuindo com peso equivalente.
- **Imunidade a escala na Arvore de Decisao:** Algoritmos baseados em arvores nao calculam distancias entre multiplos atributos simultaneamente. Em cada no, a Arvore de Decisao busca um limiar de corte $x_j \le t$ em uma unica variavel por vez que maximize a reducao da impureza de Gini. Como o `StandardScaler` aplica uma transformacao linear monotona ($z = (x - \mu)/\sigma$), a ordenacao dos valores nao se altera: um ponto de corte em `torque_nm <= 46.40` divide exatamente os mesmos registros que o ponto de corte equivalente na variavel padronizada, gerando a mesma estrutura de nos e a mesma acuracia. Manter os dados sem escalonar para a Arvore preserva a leitura direta das regras fisicas em Kelvin, RPM e Newton-metro.

## Fase 6 - Modelagem e Analise de Overfitting

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score

resultados = []

valores_k = [3, 5, 7, 9]
for k in valores_k:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_knn, y_train_bal)
    pred_treino = knn.predict(X_train_knn)
    pred_teste = knn.predict(X_test_knn)
    acc_tr = accuracy_score(y_train_bal, pred_treino)
    acc_te = accuracy_score(y_test, pred_teste)
    rec_te = recall_score(y_test, pred_teste)
    f1_te = f1_score(y_test, pred_teste)
    resultados.append({
        "Modelo": "KNN",
        "Hiperparametro": f"n_neighbors={k}",
        "Acuracia_Treino": round(acc_tr, 4),
        "Acuracia_Teste": round(acc_te, 4),
        "Diferenca_Treino_Teste": round(acc_tr - acc_te, 4),
        "Recall_Teste": round(rec_te, 4),
        "F1_Teste": round(f1_te, 4)
    })

profundidades = [3, 5, 10, None]
for d in profundidades:
    arvore = DecisionTreeClassifier(max_depth=d, random_state=42)
    arvore.fit(X_train_bal, y_train_bal)
    pred_treino = arvore.predict(X_train_bal)
    pred_teste = arvore.predict(X_test)
    acc_tr = accuracy_score(y_train_bal, pred_treino)
    acc_te = accuracy_score(y_test, pred_teste)
    rec_te = recall_score(y_test, pred_teste)
    f1_te = f1_score(y_test, pred_teste)
    resultados.append({
        "Modelo": "Arvore de Decisao",
        "Hiperparametro": f"max_depth={d}",
        "Acuracia_Treino": round(acc_tr, 4),
        "Acuracia_Teste": round(acc_te, 4),
        "Diferenca_Treino_Teste": round(acc_tr - acc_te, 4),
        "Recall_Teste": round(rec_te, 4),
        "F1_Teste": round(f1_te, 4)
    })

df_resultados = pd.DataFrame(resultados)
df_resultados